In [1]:
import pandas as pd
import glob


In [11]:
import glob
import pandas as pd

PATH = r"C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw/"

files = glob.glob(PATH + "CRMLSListing*")

print(f"Number of monthly files found: {len(files)}")

row_counts = []
for f in files:
    temp = pd.read_csv(f, encoding='latin1')
    row_counts.append((f, len(temp)))

print("\nRow counts BEFORE concatenation:")
for fname, count in row_counts:
    print(f"{fname}: {count:,} rows")

df_listed = pd.concat(
    [pd.read_csv(f, encoding='latin1') for f in files],
    ignore_index=True
)

print(f"\nTotal rows AFTER concatenation: {len(df_listed):,}")

df_res = df_listed[df_listed["PropertyType"] == "Residential"]

print(f"Total rows AFTER Residential filter: {len(df_res):,}")

df_res.to_csv("combined_listed.csv", index=False)
print("\nSaved filtered dataset to combined_listed.csv")


Number of monthly files found: 27

Row counts BEFORE concatenation:
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSListing202401.csv: 27,454 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSListing202402.csv: 27,447 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSListing202403.csv: 32,282 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSListing202404.csv: 36,503 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSListing202405.csv: 38,796 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSListing202406.csv: 35,893 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSListing202407.csv: 36,340 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSListing202408.csv: 35,305 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSListing202409.csv: 34,625 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSListing202410.csv: 34,730 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSListin

In [14]:
df_res.shape
df_res.info()


<class 'pandas.core.frame.DataFrame'>
Index: 534610 entries, 2 to 845400
Data columns (total 84 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   OriginalListPrice             533842 non-null  float64
 1   ListingKey                    534610 non-null  int64  
 2   ListAgentEmail                493182 non-null  object 
 3   CloseDate                     169865 non-null  object 
 4   ClosePrice                    150269 non-null  float64
 5   ListAgentFirstName            530455 non-null  object 
 6   ListAgentLastName             534570 non-null  object 
 7   Latitude                      454543 non-null  float64
 8   Longitude                     454543 non-null  float64
 9   UnparsedAddress               533969 non-null  object 
 10  PropertyType                  534610 non-null  object 
 11  LivingArea                    534071 non-null  float64
 12  ListPrice                     534610 non-null  fl

In [17]:

print("Initial row count:", len(df_res))
print("Initial column count:", df_res.shape[1])

# 1. DOCUMENT UNIQUE PROPERTY TYPES
print("\nUnique Property Types:")
print(df_listed["PropertyType"].unique())

# 2. NULL COUNT SUMMARY TABLE
null_summary = df_res.isnull().sum().to_frame(name="NullCount")
null_summary["NullPercent"] = (null_summary["NullCount"] / len(df_res)) * 100

print("\nNull Count Summary Table:")
print(null_summary)

# 3. FLAG COLUMNS ABOVE 90% NULL
high_null_cols = null_summary[null_summary["NullPercent"] > 90].index.tolist()

print("\nColumns ABOVE 90% null:")
for col in high_null_cols:
    print(f"- {col}")

# 4. REMOVE COLUMNS ABOVE 90% NULL
df_filtered = df_res.drop(columns=high_null_cols)
print(f"\nColumn count AFTER removing >90% null columns: {df_filtered.shape[1]}")

# 5. NUMERIC DISTRIBUTION SUMMARY
#    For ClosePrice, LivingArea, DaysOnMarket
numeric_cols = ["ClosePrice", "LivingArea", "DaysOnMarket"]

print("\nNumeric Distribution Summary:")
for col in numeric_cols:
    if col in df_filtered.columns:
        print(f"\n--- {col} ---")
        print(df_filtered[col].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))
    else:
        print(f"\n--- {col} NOT FOUND IN DATASET ---")

# 6. SAVE FILTERED DATASET
df_filtered.to_csv("filtered_week2_3_listed_output.csv", index=False)
print("\nSaved cleaned dataset to filtered_week2_3_listed_output.csv")


Initial row count: 534610
Initial column count: 84

Unique Property Types:
['ManufacturedInPark' 'CommercialSale' 'Residential' 'ResidentialLease'
 'Land' 'ResidentialIncome' 'CommercialLease' 'BusinessOpportunity']

Null Count Summary Table:
                              NullCount  NullPercent
OriginalListPrice                   768     0.143656
ListingKey                            0     0.000000
ListAgentEmail                    41428     7.749200
CloseDate                        364745    68.226371
ClosePrice                       384341    71.891846
...                                 ...          ...
BuyerOfficeName.1                373506    69.865135
AssociationFee                   127947    23.932773
LotSizeSquareFeet                 43411     8.120125
MiddleOrJuniorSchoolDistrict     534610   100.000000
UnparsedAddress.1                   641     0.119900

[84 rows x 2 columns]

Columns ABOVE 90% null:
- FireplacesTotal
- AboveGradeFinishedArea
- TaxAnnualAmount
- BuilderNam